# Fama-French Regressions by Year and Decile
Este notebook recalcula as regressões de Fama-French de 3 Fatores para as carteiras (decis) formadas pelas métricas de rede (HRM e Pozzi).
A lógica atual utiliza os ativos agrupados em 10 decis para cada ano, construídos _In-Sample_ (ano $t$), e avaliados _Out-of-Sample_ (ano $t+1$).\n

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)

## 1. Processamento e Regressões OLS
Aqui nós iteramos pelos anos (2014 a 2024), carregamos as rentabilidades diárias do ano seguinte, filtramos os tickers de cada decil e rodamos a regressão OLS.
Usamos o cálculo de **Equal-Weighted** log returns para os portfólios, garantindo que a carteira capte o efeito puramente estrutural da rede (sem enviesar por Market Cap).\n

In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

# Carrega os fatores de Fama-French
factors_df = pd.read_parquet("../../data/02_clean/fama_french_factors.parquet")

# Os fatores originais geralmente vêm em porcentagem (ex: 1.5%), então dividimos por 100
factors_df = factors_df / 100

years = range(2014, 2025)
# Alterado para 'hcm', mas se o seu arquivo ainda for 'hrm', basta mudar aqui!
metrics = ['hcm', 'pozzi'] 
deciles = [f'decil_{i}' for i in range(1, 11)]

# Dicionário para armazenar resultados puros das regressões
regression_results = []

for metric in metrics:
    print(f"Processando regressões para: {metric.upper()}...")
    
    # Carrega metadados que dizem qual Ticker está em qual decil a cada ano
    try:
        df_meta = pd.read_parquet(f"../../data/07_portfolios_metadata/complete_metadata_{metric}.parquet")
        df_meta['year'] = df_meta['year'].astype(int)
    except FileNotFoundError:
        print(f"Arquivo de metadados para {metric} não encontrado. Pule.")
        continue
    
    for year in years:
        # Puxa retornos Out-of-Sample (ano + 1)
        try:
            oos_ret = pd.read_parquet(f"../../data/02_clean/returns_new_{year+1}.parquet")
        except FileNotFoundError:
            continue
            
        # Alinha os fatores de Fama-French às datas dos retornos daquele ano
        factors_year = factors_df[(factors_df.index >= oos_ret.index[0]) & (factors_df.index <= oos_ret.index[-1])]
        
        # Dicionário temporário para guardar os retornos das pontas do PMC
        port_returns_year = {}
        
        # =========================================================
        # 1. Regressões individuais para cada decil
        # =========================================================
        for decil in deciles:
            # Puxa tickers daquele decil no ano 'year'
            tickers = df_meta[(df_meta['year'] == year) & (df_meta['portfolio'] == decil)]['Ticker'].tolist()
            
            # Garante que as ações existam na base de retornos do ano t+1
            valid_tickers = [t for t in tickers if t in oos_ret.columns]
            
            if len(valid_tickers) == 0:
                continue
                
            # Calcula retorno Equal-Weighted da carteira
            port_ret = np.log1p(oos_ret[valid_tickers]).mean(axis=1)
            
            # Salva para criar o PMC depois
            if decil in ['decil_1', 'decil_10']:
                port_returns_year[decil] = port_ret
            
            # Subtrai a Risk-Free rate para obter o Excesso de Retorno
            excess_ret = port_ret - factors_year['RF']
            
            # Variáveis independentes
            X = factors_year[['Mkt-RF', 'SMB', 'HML']]
            X = sm.add_constant(X)
            
            # Alinhamento por data e drop de NAs
            aligned = pd.concat([excess_ret.rename("ExRet"), X], axis=1, join="inner").dropna()
            
            if len(aligned) < 30:
                continue
                
            # Roda a Regressão
            model = sm.OLS(aligned['ExRet'], aligned[['const', 'Mkt-RF', 'SMB', 'HML']]).fit()
            
            regression_results.append({
                'metric': metric,
                'year_oos': year + 1,
                'year_formed': year,
                'decil': decil,
                'alpha': model.params['const'] * 252, # Alpha Anualizado
                'alpha_tstat': model.tvalues['const'],
                'mkt_beta': model.params['Mkt-RF'],
                'smb_beta': model.params['SMB'],
                'hml_beta': model.params['HML'],
                'r_squared': model.rsquared
            })
            
        # =========================================================
        # 2. Regressão do Portfólio PMC (Peripheral Minus Central)
        # =========================================================
        if 'decil_10' in port_returns_year and 'decil_1' in port_returns_year:
            # Retorno do PMC = Peripheral(decil_10) - Central(decil_1)
            pmc_ret = port_returns_year['decil_10'] - port_returns_year['decil_1']
            
            # NOTA: O fator RF se anula em carteiras Long-Short, 
            # portanto o retorno líquido pmc_ret JÁ É O EXCESSO DE RETORNO!
            
            X = factors_year[['Mkt-RF', 'SMB', 'HML']]
            X = sm.add_constant(X)
            
            aligned_pmc = pd.concat([pmc_ret.rename("ExRet"), X], axis=1, join="inner").dropna()
            
            if len(aligned_pmc) >= 30:
                model_pmc = sm.OLS(aligned_pmc['ExRet'], aligned_pmc[['const', 'Mkt-RF', 'SMB', 'HML']]).fit()
                
                regression_results.append({
                    'metric': metric,
                    'year_oos': year + 1,
                    'year_formed': year,
                    'decil': 'PMC (10 - 1)', # Identificador único para essa carteira
                    'alpha': model_pmc.params['const'] * 252,
                    'alpha_tstat': model_pmc.tvalues['const'],
                    'mkt_beta': model_pmc.params['Mkt-RF'],
                    'smb_beta': model_pmc.params['SMB'],
                    'hml_beta': model_pmc.params['HML'],
                    'r_squared': model_pmc.rsquared
                })

df_results = pd.DataFrame(regression_results)
print("Todas as regressões foram calculadas, incluindo a carteira PMC!")

Processando regressões para: HCM...
Processando regressões para: POZZI...
Todas as regressões foram calculadas, incluindo a carteira PMC!


In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

# Carrega os fatores de Fama-French
factors_df = pd.read_parquet("../../data/02_clean/fama_french_factors.parquet")

# Os fatores originais geralmente vêm em porcentagem (ex: 1.5%), então dividimos por 100
factors_df = factors_df / 100

years = range(2014, 2025)
# Alterado para 'hcm', mas se o seu arquivo ainda for 'hrm', basta mudar aqui!
metrics = ['hcm', 'pozzi'] 
deciles = [f'decil_{i}' for i in range(1, 11)]

# Dicionário para armazenar resultados puros das regressões
regression_results = []

for metric in metrics:
    print(f"Processando regressões para: {metric.upper()}...")
    
    # Carrega metadados que dizem qual Ticker está em qual decil a cada ano
    try:
        df_meta = pd.read_parquet(f"../../data/07_portfolios_metadata/complete_metadata_{metric}.parquet")
        df_meta['year'] = df_meta['year'].astype(int)
    except FileNotFoundError:
        print(f"Arquivo de metadados para {metric} não encontrado. Pule.")
        continue
    
    for year in years:
        # Puxa retornos Out-of-Sample (ano + 1)
        try:
            oos_ret = pd.read_parquet(f"../../data/02_clean/returns_new_{year+1}.parquet")
        except FileNotFoundError:
            continue
            
        # Alinha os fatores de Fama-French às datas dos retornos daquele ano
        factors_year = factors_df[(factors_df.index >= oos_ret.index[0]) & (factors_df.index <= oos_ret.index[-1])]
        
        # --- CONVERSÃO PARA RETORNOS MENSAIS ---
        # Agrupa por final do mês ('ME' ou 'M') e compõe os retornos: (1 + r).prod() - 1
        # Usamos 'ME' que é o novo padrão do pandas para Month End
        oos_ret_monthly = (1 + oos_ret).resample('ME').prod() - 1
        factors_monthly = (1 + factors_year).resample('ME').prod() - 1
        
        # Dicionário temporário para guardar os retornos das pontas do PMC
        port_returns_year = {}
        
        # 1. Regressões individuais para cada decil
        for decil in deciles:
            # Puxa tickers daquele decil no ano 'year'
            tickers = df_meta[(df_meta['year'] == year) & (df_meta['portfolio'] == decil)]['Ticker'].tolist()
            
            # Garante que as ações existam na base de retornos mensais
            valid_tickers = [t for t in tickers if t in oos_ret_monthly.columns]
            
            if len(valid_tickers) == 0:
                continue
                
            # Calcula retorno Equal-Weighted da carteira baseado nos retornos mensais
            port_ret = np.log1p(oos_ret_monthly[valid_tickers]).mean(axis=1)
            
            # Salva para criar o PMC depois
            if decil in ['decil_1', 'decil_10']:
                port_returns_year[decil] = port_ret
            
            # Subtrai a Risk-Free rate para obter o Excesso de Retorno
            excess_ret = port_ret - factors_monthly['RF']
            
            # Variáveis independentes mensais
            X = factors_monthly[['Mkt-RF', 'SMB', 'HML']]
            X = sm.add_constant(X)
            
            # Alinhamento por data e drop de NAs
            aligned = pd.concat([excess_ret.rename("ExRet"), X], axis=1, join="inner").dropna()
            
            # Alterado de 30 para 10, já que agora temos no máximo 12 observações (meses) no ano
            if len(aligned) < 10:
                continue
                
            # Roda a Regressão
            model = sm.OLS(aligned['ExRet'], aligned[['const', 'Mkt-RF', 'SMB', 'HML']]).fit()
            
            regression_results.append({
                'metric': metric,
                'year_oos': year + 1,
                'year_formed': year,
                'decil': decil,
                'alpha': model.params['const'] * 12, # Alpha Anualizado (12 meses ao invés de 252 dias)
                'alpha_tstat': model.tvalues['const'],
                'mkt_beta': model.params['Mkt-RF'],
                'smb_beta': model.params['SMB'],
                'hml_beta': model.params['HML'],
                'r_squared': model.rsquared
            })
            
        # 2. Regressão do Portfólio PMC (Peripheral Minus Central)
        if 'decil_10' in port_returns_year and 'decil_1' in port_returns_year:
            # Retorno do PMC mensal = Peripheral(decil_10) - Central(decil_1)
            pmc_ret = port_returns_year['decil_10'] - port_returns_year['decil_1']
            
            X = factors_monthly[['Mkt-RF', 'SMB', 'HML']]
            X = sm.add_constant(X)
            
            aligned_pmc = pd.concat([pmc_ret.rename("ExRet"), X], axis=1, join="inner").dropna()
            
            # Alterado de 30 para 10
            if len(aligned_pmc) >= 10:
                model_pmc = sm.OLS(aligned_pmc['ExRet'], aligned_pmc[['const', 'Mkt-RF', 'SMB', 'HML']]).fit()
                
                regression_results.append({
                    'metric': metric,
                    'year_oos': year + 1,
                    'year_formed': year,
                    'decil': 'PMC (10 - 1)', 
                    'alpha': model_pmc.params['const'] * 12, # Alpha Anualizado (12 meses)
                    'alpha_tstat': model_pmc.tvalues['const'],
                    'mkt_beta': model_pmc.params['Mkt-RF'],
                    'smb_beta': model_pmc.params['SMB'],
                    'hml_beta': model_pmc.params['HML'],
                    'r_squared': model_pmc.rsquared
                })

df_results = pd.DataFrame(regression_results)
print("Todas as regressões foram calculadas, incluindo a carteira PMC!")

Processando regressões para: HCM...
Processando regressões para: POZZI...
Todas as regressões foram calculadas, incluindo a carteira PMC!


## 2. Opção de Tabela: Resumo Estilo Fama-MacBeth
Esta tabela sintetiza 10 anos de regressões. Ela exibe a **média temporal** dos Alphas e dos Betas para cada decil. 
O teste t (Alpha_tstat_FM) é calculado dividindo a média do Alpha pelo Erro Padrão da média (Desvio Padrão do Alpha / raiz do número de anos). Essa é a forma mais clássica de relatar performance em asset pricing.\n

In [3]:
def generate_fama_macbeth_table(df_metric_results, metric_name):
    if df_metric_results.empty:
        return None
        
    # Número de anos na amostra OOS
    n_years = df_metric_results['year_oos'].nunique()
    
    # Agrega tirando a média das variáveis e o desvio padrão do alpha
    # Como já incluímos o "PMC (10 - 1)" nas regressões originais, 
    # ele já entra neste cálculo automático junto com os decis!
    summary = df_metric_results.groupby('decil').agg({
        'alpha': ['mean', 'std'],
        'mkt_beta': 'mean',
        'smb_beta': 'mean',
        'hml_beta': 'mean',
        'r_squared': 'mean'
    })
    
    # Calcula T-Statistic no estilo Fama-MacBeth (Mean / Standard Error)
    summary['Alpha_tstat_FM'] = summary[('alpha', 'mean')] / (summary[('alpha', 'std')] / np.sqrt(n_years))
    
    # Achata as colunas (Flatten)
    summary.columns = ['Alpha', 'Alpha_Std', 'Mkt_Beta', 'SMB_Beta', 'HML_Beta', 'R_Squared', 'Alpha_tstat_FM']
    
    # Reordena para ficar bonito (decil_1 até decil_10 e depois a carteira Long-Short PMC)
    decil_order = [f'decil_{i}' for i in range(1, 11)]
    if 'PMC (10 - 1)' in summary.index:
        decil_order.append('PMC (10 - 1)')
    
    # Filtra e aplica a ordem correta
    summary = summary.reindex(decil_order)
    
    # Formata para visualização
    final_table = summary[['Alpha', 'Alpha_tstat_FM', 'Mkt_Beta', 'SMB_Beta', 'HML_Beta', 'R_Squared']].copy()
    final_table = final_table.round(4)
    
    # Renomeia o índice linha por linha com base no nome original
    index_names = []
    for idx in final_table.index:
        if idx == 'decil_1':
            index_names.append('Decil 1 (Central)')
        elif idx == 'decil_10':
            index_names.append('Decil 10 (Peripheral)')
        elif idx == 'PMC (10 - 1)':
            index_names.append('Long-Short (Peripheral - Central)')
        else:
            index_names.append(idx.replace('_', ' ').title())
            
    final_table.index = index_names
    
    print(f"\n== FAMA-MACBETH REGRESSION SUMMARY: {metric_name.upper()} ===")
    display(final_table)
    
    return final_table

# Executa para HCM e Pozzi (lembrando que atualizamos a nomenclatura!)
fm_hcm = generate_fama_macbeth_table(df_results[df_results['metric'] == 'hcm'], 'hcm')
fm_pozzi = generate_fama_macbeth_table(df_results[df_results['metric'] == 'pozzi'], 'pozzi')

# Salva tabelas
if fm_hcm is not None:
    fm_hcm.to_csv("../../data/07_portfolios_metadata/table_famamacbeth_hcm.csv")
if fm_pozzi is not None:
    fm_pozzi.to_csv("../../data/07_portfolios_metadata/table_famamacbeth_pozzi.csv")


== FAMA-MACBETH REGRESSION SUMMARY: HCM ===


,Alpha,Alpha_tstat_FM,Mkt_Beta,SMB_Beta,HML_Beta,R_Squared
Decil 1 (Central),-0.0803,-5.4008,0.9684,0.5319,0.4365,0.9640
Decil 2,-0.0477,-3.3865,0.9453,0.5185,0.3487,0.9664
Decil 3,-0.0623,-4.1275,0.9510,0.4910,0.3926,0.9633
Decil 4,-0.0712,-5.5800,0.9566,0.4415,0.3159,0.9717
Decil 5,-0.0684,-4.9983,0.8926,0.4311,0.2487,0.9577
Decil 6,-0.0635,-5.5081,0.8694,0.4272,0.2214,0.9244
Decil 7,-0.0661,-3.2497,0.8175,0.3183,0.0973,0.8966
Decil 8,-0.1098,-6.9099,0.8246,0.4369,0.1326,0.8710
Decil 9,-0.1412,-4.1320,0.7124,0.3842,0.0932,0.8214
Decil 10 (Peripheral),-0.0873,-2.4511,0.4999,0.3100,0.0192,0.6919



== FAMA-MACBETH REGRESSION SUMMARY: POZZI ===


,Alpha,Alpha_tstat_FM,Mkt_Beta,SMB_Beta,HML_Beta,R_Squared
Decil 1 (Central),-0.0622,-3.4534,0.8777,0.2568,0.3477,0.9131
Decil 2,-0.0412,-2.4531,0.8147,0.2375,0.2943,0.8796
Decil 3,-0.0637,-3.4027,0.8223,0.3566,0.2490,0.9439
Decil 4,-0.0494,-3.6942,0.9404,0.4076,0.2454,0.8921
Decil 5,-0.0756,-3.1130,0.8955,0.5102,0.1905,0.8839
Decil 6,-0.0687,-3.2912,0.8570,0.4682,0.2136,0.9198
Decil 7,-0.0931,-4.0378,0.8311,0.6032,0.1632,0.9491
Decil 8,-0.0789,-3.2144,0.7892,0.5275,0.3129,0.9278
Decil 9,-0.1175,-5.7355,0.7858,0.3897,0.1382,0.8590
Decil 10 (Peripheral),-0.1474,-3.6337,0.8236,0.5335,0.1512,0.8290
